In [7]:
import pandas as pd
import numpy as np
import pyreadr


# Modelling
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score,make_scorer, root_mean_squared_error, ConfusionMatrixDisplay
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import GridSearchCV, KFold
from scipy.stats import randint
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import sklearn
from copy import deepcopy
from xgboost import XGBRegressor



from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_error

# Models
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor


In [2]:

train:pd.DataFrame = pyreadr.read_r('datasets/train.rds')[None]
rs_seed=1


In [3]:

def Search(model,
           par_grid:dict,
           df:pd.DataFrame,
           scale=True,
           rs_seed:int=1) -> sklearn.model_selection.GridSearchCV:
    """
    Universal GridSearchCV function for regression models with preprocessing.

    Parameters
    ----------
    model : sklearn estimator
        The model to tune (e.g. RandomForestRegressor()).
    par_grid : dict
        Parameter grid for GridSearchCV (keys must include 'Model__').
    df : pd.DataFrame
        Input DataFrame containing predictors + 'score' column as target.
    scale : bool, default=True
        Whether to scale numerical features.
    rs_seed : int, default=1
        Random state for reproducibility.

    Returns
    -------
    search : GridSearchCV object (fitted)
    """

    # Create a deeocopy because i am scared of references
    X = deepcopy(df.drop(columns=["score"]))
    y = deepcopy(df["score"])

    cats = X.select_dtypes(include=["object", "category"]).columns.tolist()
    nums = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

    preprocessor = ColumnTransformer([
        ("categorical", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), cats),
        ("numeric", StandardScaler() if scale else "passthrough", nums)
    ])

    scores = {
        "RMSE": make_scorer(root_mean_squared_error, greater_is_better=False),
        "R2": make_scorer(r2_score, greater_is_better=True)
    }

    pipe = Pipeline([
        ("Preprocess", preprocessor),
        ("Model", model)
    ])

    search = GridSearchCV(
        estimator=pipe,
        param_grid=par_grid,
        cv=KFold(n_splits=5, shuffle=True, random_state=rs_seed),
        scoring=scores,
        refit="RMSE",
        n_jobs=-1,
        verbose=2,
        return_train_score=True,
    )
    search.fit(X, y)
    return search


In [ ]:
param_rf = {
    "Model__n_estimators": np.arange(400, 1201, 40),
    "Model__max_depth": [None] + list(np.arange(3, 9, 1)),
    "Model__min_samples_split": np.arange(2, 21, 1),
    "Model__min_samples_leaf": np.arange(8, 16, 1),
    "Model__max_features": ["sqrt", 0.3, 1.0],
    "Model__max_leaf_nodes": [None] + list(np.arange(40, 201, 5)),
    "Model__min_impurity_decrease": np.linspace(0, 1e-8, 30),
    "Model__ccp_alpha": np.linspace(0, 0.02, 20)
}

rf = RandomForestRegressor(bootstrap=True,
                           criterion='squared_error',
                           random_state=rs_seed)

rf_search = Search(
    model=rf,
    par_grid = param_rf,
    df = train)

In [ ]:
xgb_param = {
    "Model__n_estimators": np.arange(400, 1201, 40),
    "Model__learning_rate": np.linspace(0.01, 0.2, 40),
    "Model__max_depth": np.arange(2, 9, 1),
    "Model__subsample": np.linspace(0.5, 1, 20),
    "Model__colsample_bytree": np.linspace(0.5, 1, 20),
    "Model__reg_lambda": np.logspace(-3, 1, 20), # from ~0 to 10
    "Model__reg_alpha": np.logspace(-3, 1, 20),
    "Model__min_child_weight": np.arange(2, 9, 1),
    "Model__gamma": np.linspace(0, 5, 50),
    "Model__grow_policy": ["lossguide", "depthwise"]
}


xgb = XGBRegressor(
    random_state=rs_seed
)

xgb_search = Search(
    model=xgb,
    par_grid=xgb_param,
    df=train
)


In [ ]:
dtr_param = {
    "Model__criterion": ["squared_error", "friedman_mse"],
    "Model__max_depth": np.arange(10, 26, 1),
    "Model__min_samples_split": np.arange(5, 26, 1),
    "Model__min_samples_leaf": np.arange(5, 19, 1),
    "Model__max_features": ["sqrt", 0.3, 0.5, 0.7, 1.0],
    "Model__ccp_alpha": np.linspace(0, 0.05, 20),
    "Model__min_impurity_decrease": np.linspace(0, 0.01, 50),
}

dtr=DecisionTreeRegressor(splitter="best",
                          max_leaf_nodes=False,
                          random_state=rs_seed)


dtr_search = Search(
    model=dtr,
    par_grid=dtr_param,
    df=train
)